# Community Event Classification

This notebook classifies lifecycle events between every pair of consecutive snapshots for cumulative, interval, and overlapping graphs.

- `birth`: zero-to-one
- `death`: one-to-zero
- `continuation`: exclusive one-to-one
- `split`: one-to-many
- `merge`: many-to-one

An event relationship is accepted when either prospective or retrospective stability is at least `0.4`. Observation IDs are snapshot-scoped, while final community labels are assigned after the whole lifespan is assembled.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "graph-matching":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "graph-matching"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from stage_events import run_approach

INPUT_DIR = NOTEBOOK_DIR / "outputs" / "graph_matching"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "stage_identification"
APPROACHES = ["cumulative", "interval", "overlap"]
JACCARD_THRESHOLD = 0.0
STABILITY_THRESHOLD = 0.4


## Generate Event Tables

In [ ]:
for approach in APPROACHES:
    run_approach(
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        approach=approach,
        jaccard_threshold=JACCARD_THRESHOLD,
        stability_threshold=STABILITY_THRESHOLD,
    )

print(f"Event tables written to {OUTPUT_DIR}")

## Event Counts

In [ ]:
event_tables = {
    approach: pd.read_csv(OUTPUT_DIR / approach / "community_events.csv", keep_default_na=False)
    for approach in APPROACHES
}

event_counts = (
    pd.concat(event_tables.values(), ignore_index=True)
    .groupby(["approach", "event_type"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["birth", "death", "continuation", "split", "merge"], fill_value=0)
)
event_counts


Event cardinality uses directional stability rather than Jaccard alone. This preserves split branches that retain a substantial share of the parent and merge branches that retain a substantial share of the child.

The first events are usually initial births, so their source and overlap columns are intentionally blank. The inspection cell below shows major events first to avoid a wall of empty birth fields.


In [ ]:
approach_to_inspect = "interval"
events = event_tables[approach_to_inspect]
major_events = events[events["event_type"].isin(["split", "merge"])]
major_events.head(15)
